# QWEN-999B: Core Training Pipeline on Google Colab / Kaggle (T4 15 GB)
**Objective:** Train QWEN-999B using Unsloth 4-bit QLoRA on a free T4 GPU (15 GB VRAM) with Zero-OOM guarantee.

### Guardrails & Constraints Applied:
- **Zero-OOM T4 Hardware Profile:** `load_in_4bit=True`, `max_seq_length=2048`, `per_device_train_batch_size=2`, `gradient_accumulation_steps=4`, `optim="adamw_8bit"`
- **Precision Constraint:** `fp16=True`, `bf16=False` (Strict: NVIDIA T4 lacks native hardware BF16 compute)
- **Dataset Ingestion:** Direct streaming/pulling from `HF_DATASET_REPO` using Colab Secrets (`userdata.get`) or environment variables
- **Train/Eval Split:** 95% training, 5% validation with step-wise loss evaluation (`eval_steps=50`, `logging_steps=5`)
- **Checkpoint Resilience:** `save_strategy='steps'`, `save_steps=50`, `save_total_limit=3`, `resume_from_checkpoint=True`, and auto-push to Hugging Face
- **Ollama Export:** Q4_K_M GGUF export for local inference on Ryzen PC / Dell 15

### 1. Environment Setup & Dependency Installation
Install Unsloth, PyTorch, Transformers, and TRL optimized for NVIDIA T4.

In [2]:
# Install Unsloth and compatible libraries
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7cuba56d/unsloth_43b5b276fe074b4987227bbb52cb3c12
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7cuba56d/unsloth_43b5b276fe074b4987227bbb52cb3c12
ERROR: Operation cancelled by user


### 2. Verify GPU & Hardware Profile
Ensure an NVIDIA T4 or P100 with at least 15 GB VRAM is active.

In [3]:
import torch
assert torch.cuda.is_available(), "CUDA GPU is required for training! Please change runtime to T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Connected to: {gpu_name} with {vram_gb:.2f} GB VRAM")

Connected to: Tesla T4 with 14.56 GB VRAM


### 3. Hugging Face Authentication & Secrets Setup
Reads `HF_TOKEN` and `HF_DATASET_REPO` from Colab Secrets (`userdata.get`) or environment variables (`os.environ`). Never hardcode tokens in notebooks.

In [5]:
import os
from huggingface_hub import login

# Read HF_TOKEN and HF_DATASET_REPO from Colab Secrets with environment fallback
hf_token = None
hf_dataset_repo = None

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN').strip()
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

try:
    from google.colab import userdata
    hf_dataset_repo = userdata.get('HF_DATASET_REPO').strip()
except Exception:
    hf_dataset_repo = os.environ.get('HF_DATASET_REPO')

if hf_token:
    login(token=hf_token)
    print("Authenticated successfully with Hugging Face.")
else:
    print("Warning: HF_TOKEN not set. Set HF_TOKEN in Colab Secrets (key icon) or export HF_TOKEN to enable auto-checkpoint sync to Hub.")

if hf_dataset_repo:
    print(f"Target Hugging Face Dataset Repo: {hf_dataset_repo}")
else:
    print("Warning: HF_DATASET_REPO not set. Set HF_DATASET_REPO in Colab Secrets or environment to pull the remote master dataset.")

Authenticated successfully with Hugging Face.
Target Hugging Face Dataset Repo: mwankhede/qwen-9999b-core-v1


### 4. Load Base Model in 4-bit with FastLanguageModel
We use `unsloth/Qwen2.5-7B-Instruct` with strict Zero-OOM parameters: `load_in_4bit=True` and `max_seq_length=2048`.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection (fp16 on T4)
load_in_4bit = True
model_name = "unsloth/Qwen2.5-7B-Instruct"  # Core generalist foundation

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

### 5. Attach PEFT LoRA Adapters
Per `unsloth-t4-trainer` specs:
- `r = 16`, `lora_alpha = 32`
- Target all 7 linear projections: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- `use_gradient_checkpointing = 'unsloth'`

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA adapter attached successfully.")

### 6. Ingest Master Dataset & Split into Train / Eval
Pulls directly from `HF_DATASET_REPO` via Hugging Face Hub (with local and starter fallbacks).
Standardizes into Qwen ChatML format (`<think>...</think>`) and splits into 95% training and 5% validation sets.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset

# Configure official Qwen 2.5 ChatML template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# Priority 1: Pull directly from HF_DATASET_REPO using Colab Secrets / Environment
raw_dataset = None
if hf_dataset_repo:
    print(f"Pulling master dataset directly from Hugging Face: {hf_dataset_repo}...")
    try:
        raw_dataset = load_dataset(hf_dataset_repo, split="train", token=hf_token)
        print(f"Successfully loaded {len(raw_dataset)} samples from {hf_dataset_repo}")
    except Exception as e:
        print(f"Failed to load from HF_DATASET_REPO '{hf_dataset_repo}': {e}. Falling back...")

# Priority 2: Fall back to local file if available
if raw_dataset is None:
    dataset_file = "data/reasoning_data.jsonl"
    if os.path.exists(dataset_file):
        print(f"Loading local dataset from {dataset_file}...")
        raw_dataset = load_dataset("json", data_files=dataset_file, split="train")
        print(f"Loaded {len(raw_dataset)} samples from {dataset_file}")

# Priority 3: Fall back to embedded starter dataset
if raw_dataset is None:
    print("Using embedded starter reasoning dataset with <think> traces...")
    starter_samples = [
        {
            "messages": [
                {"role": "system", "content": "You are QWEN-999B, a high-reasoning AI assistant specialized in logic, coding, and multilingual problem-solving."},
                {"role": "user", "content": "How many r's are in the word strawberry?"},
                {"role": "assistant", "content": "<think>\nAnalyze the word 'strawberry' letter by letter:\ns - 1\nt - 2\nr - 3 (first 'r')\na - 4\nw - 5\nb - 6\ne - 7\nr - 8 (second 'r')\nr - 9 (third 'r')\ny - 10\nTotal occurrences of 'r': 3.\n</think>\nThere are **3** \"r\"s in the word **strawberry**."}
            ]
        },
        {
            "messages": [
                {"role": "system", "content": "You are QWEN-999B, a high-reasoning AI assistant specialized in logic, coding, and multilingual problem-solving."},
                {"role": "user", "content": "Bhai explain karo what is Dijkstra's algorithm and iska time complexity kya hota hai?"},
                {"role": "assistant", "content": "<think>\nDijkstra's algorithm finds single-source shortest paths in weighted graphs with non-negative edge weights.\nUsing min-heap / priority queue:\nTime complexity: O((V + E) log V).\nUser asked in Hinglish, respond with friendly, precise technical Hinglish.\n</think>\nBhai **Dijkstra's Algorithm** ek greedy graph algorithm hai jo single source se graph ke baaki saare nodes ka **shortest path** nikalta hai.\n\n### Key Rules:\n1. **Non-negative weights:** Yeh algorithm sirf non-negative edge weights par kaam karta hai (negative weights ke liye Bellman-Ford use karte hain).\n2. **Min-Heap / Priority Queue:** Current minimum distance wale node ko explore karne ke liye min-heap use hota hai.\n\n### Time & Space Complexity:\n- **Time Complexity:** `O((V + E) log V)` (jab min-heap use karein, jahan V = vertices aur E = edges).\n- **Space Complexity:** `O(V)` distances aur visited tracking ke liye."}
            ]
        }
    ] * 50
    raw_dataset = Dataset.from_list(starter_samples)
    print(f"Initialized starter dataset with {len(raw_dataset)} samples.")

# Format samples into ChatML text
formatted_dataset = raw_dataset.map(formatting_prompts_func, batched=True)

# Implements strict 95% train / 5% validation split
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Dataset split complete: {len(train_dataset)} training samples, {len(eval_dataset)} validation samples.")

### 7. Configure SFTTrainer & TrainingArguments for T4
Strict adherence to T4 hardware guardrails and resilience constraints:
- `per_device_train_batch_size = 2`
- `gradient_accumulation_steps = 4` (effective batch size 8)
- `fp16 = True`, `bf16 = False` (Mandatory: T4 lacks native hardware BF16)
- `optim = 'adamw_8bit'`
- `logging_steps = 5`
- `eval_strategy = 'steps'`, `eval_steps = 50` (clean evaluation logging on validation split)
- `save_strategy = 'steps'`, `save_steps = 50`, `save_total_limit = 3`, `push_to_hub = True`

In [ ]:
import inspect
from trl import SFTTrainer
from transformers import TrainingArguments

# Infer hub model repo id or use default
if hf_dataset_repo and "/" in hf_dataset_repo:
    user_or_org = hf_dataset_repo.split("/")[0]
    hub_model_id = f"{user_or_org}/qwen-999b-adapter"
else:
    hub_model_id = "qwen-999b-adapter"

output_dir = "outputs/qwen-999b-checkpoints"

training_args_kwargs = {
    "output_dir": output_dir,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 10,
    "max_steps": 150,               # Set to num_train_epochs or steps as needed
    "learning_rate": 2e-4,
    "fp16": True,                   # Required for NVIDIA T4
    "bf16": False,                  # STRICT: NO BF16 on T4 (hardware guardrail)
    "logging_steps": 5,             # Clean, frequent loss logging
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 3407,
    "save_strategy": "steps",
    "save_steps": 50,               # Save checkpoint every 50 steps
    "save_total_limit": 3,          # Keep last 3 checkpoints to manage disk space
    "eval_steps": 50,               # Evaluate loss every 50 steps
    "push_to_hub": bool(hf_token),  # Auto-sync checkpoints to Hub
    "hub_model_id": hub_model_id if hf_token else None,
    "report_to": "none",
}

# Determine eval_strategy vs evaluation_strategy based on transformers version
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "steps"
else:
    training_args_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_args_kwargs)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)
print("SFTTrainer configured successfully with 95/5 train/eval split and step evaluation.")

### 8. Train the Model & Log Loss
Resumes automatically from the latest checkpoint if interrupted by Colab disconnects, and evaluates validation loss.

In [ ]:
# Resume from checkpoint if interrupted
has_checkpoint = os.path.isdir(output_dir) and any(d.startswith("checkpoint-") for d in os.listdir(output_dir))
resume_checkpoint = True if has_checkpoint else False
print(f"Resume from existing checkpoint: {resume_checkpoint} (found checkpoints in {output_dir}: {has_checkpoint})")

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
print(f"Training finished! Final training loss: {train_result.training_loss:.4f}")

# Clean evaluation loss logging
eval_metrics = trainer.evaluate()
print("Final Evaluation Metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")

### 9. Save LoRA Adapters & Push to Hugging Face
Saves the trained QLoRA weights locally and uploads them to the Hugging Face Hub.

In [ ]:
final_lora_path = "outputs/qwen-999b-final-lora"
model.save_pretrained(final_lora_path)
tokenizer.save_pretrained(final_lora_path)
print(f"Saved final LoRA adapter to {final_lora_path}")

if hf_token:
    print(f"Pushing LoRA adapter to Hugging Face Hub ({hub_model_id})...")
    model.push_to_hub(hub_model_id, token=hf_token)
    tokenizer.push_to_hub(hub_model_id, token=hf_token)
    print(f"Successfully synced adapter to Hugging Face: {hub_model_id}")

### 10. Export to GGUF (Q4_K_M) for Ollama Inference
Per `gguf-ollama-packager` skill, export quantized GGUF directly for local Ryzen PC / Dell 15 Ollama execution.

In [ ]:
# Export to Q4_K_M GGUF format for Ollama compatibility
gguf_output_dir = "outputs/qwen-999b-gguf"
print("Exporting model to Q4_K_M GGUF format for Ollama...")
try:
    model.save_pretrained_gguf(
        gguf_output_dir,
        tokenizer,
        quantization_method="q4_k_m"
    )
    print(f"GGUF export completed successfully in {gguf_output_dir}!")
except Exception as e:
    print(f"GGUF export note: {e}")